In [26]:
import pandas as pd
import numpy as np
import ujson as json
from ast import literal_eval

In [37]:
df = pd.read_csv("../../data/amsterdam/tweets/amsterdam_2012-01-08_2012-01-09.csv")
df = df[df["entities"].notnull()]
df["entities"] = df["entities"].map(literal_eval)
df["mentions"] = df["entities"].map(lambda x: [mention["id"] for mention in x["mentions"]] if "mentions" in x else [])
df["mentions"]
# df["mentions"] = df["entities"].map(lambda x: x["mentions"]["id"])

/tmp/ipykernel_116196/2645844082.py:1: DtypeWarning: Columns (4,10,11,18,19,28,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../data/amsterdam/tweets/amsterdam_2012-01-08_2012-01-09.csv")


0         [274678859]
3         [319687555]
6         [294287391]
7        [2678875398]
8         [125735785]
             ...     
23803              []
23806    [1211660533]
23807    [1211660533]
23808       [2530071]
23810     [218238614]
Name: mentions, Length: 4978, dtype: object

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, BooleanType, DateType, MapType, FloatType, ArrayType
from pyspark.sql.functions import from_json, get_json_object, col, unix_timestamp, substring, udf, floor, collect_list, lit, explode
from pyspark.sql.functions import max as pyspark_max, to_timestamp, get_json_object
from pyspark.sql.functions import min as pyspark_min

from ast import literal_eval
import ujson as json

# initializing Spark session
spark = SparkSession \
    .builder\
    .config("spark.driver.memory", "50g")\
    .config("spark.jars","/mnt/common-hdd/bokanyie/postgresql-42.7.8.jar")\
    .appName("Python Spark SQL") \
    .getOrCreate()

pg_url = "jdbc:postgresql://localhost:5432/twitter_cities_test"
pg_props = {
    "user": "bokanyie",
    "password": "eCIt22X9YQHZwrWzw1JjvzB3QAI8iRSe",
    "driver": "org.postgresql.Driver"
}

# structure of tweets saved by Bence after pre-processing
tweets_schema = StructType(
    [
    StructField("attachments", StringType(), True),
    StructField("author_created_at", StringType(), True),
    StructField("author_description", StringType(), True),
    StructField("author_entitites", StringType(), True),
    StructField("author_id", LongType(), True),
    StructField("author_location", StringType(), True),
    StructField("author_name", StringType(), True),
    StructField("author_pinned_tweet_id", LongType(), True),
    StructField("author_pm_followers_count", LongType(), True),
    StructField("author_pm_following_count", LongType(), True),
    StructField("author_pm_listed_count", LongType(), True),
    StructField("author_pm_tweet_count", LongType(), True),
    StructField("author_profile_image_url", StringType(), True),
    StructField("author_protected", BooleanType(), True),
    StructField("author_url", StringType(), True),
    StructField("author_username", StringType(), True),
    StructField("author_verified", BooleanType(), True),
    StructField("author_withheld", StringType(), False),
    StructField("context_annotations", StringType(), True),
    StructField("conversation_id", LongType(), True),
    StructField("created_at", StringType(), False),
    StructField("edit_controls", StringType(), False),
    StructField("edit_history_tweet_ids", StringType(), True),
    StructField("entities", StringType(), True),
    StructField("geo_coo_coordinates", StringType(), True),
    StructField("geo_coo_type", StringType(), True),
    StructField("geo_loc_name", StringType(), True),
    StructField("geo_place_id", StringType(), True),
    StructField("id", LongType(), False),
    StructField("in_reply_to_user_id", LongType(), True),
    StructField("lang", StringType(), True),
    StructField("possibly_sensitive", BooleanType(), True),
    StructField("referenced_tweets", StringType(), True),
    StructField("reply_settings", StringType(), True),
    StructField("source", StringType(), True),    
    StructField("text", StringType(), False),
    StructField("tweet_pm_like_count", LongType(), False),
    StructField("tweet_pm_quote_count", LongType(), False),
    StructField("tweet_pm_reply_count", LongType(), False),
    StructField("tweet_pm_retweet_count", LongType(), False),
    StructField("withheld", StringType(), False)
    # StructField("corrupt_record", StringType(), True)
    ]
)
rename_city = {
    "amsterdam" : "amsterdam",
    "portland" : "portland",
    "Greater-London" : "london"
}

# for city in ["amsterdam", "portland", "Greater-London"]:
for city in ["amsterdam"]:

    # extracting mentions
    mentions = (spark.read
        .option("multiline", "true")
        .option("quote", '"')
        .option("escape", "\\")
        .option("escape", '"')
        .csv(
            # f'../../data/{city}/tweets/',
            "../../data/amsterdam/tweets/amsterdam_2012-01-08_2012-01-09.csv",
            header="True",
            schema=tweets_schema,
            mode="DROPMALFORMED",
        )
        .withColumn("city", lit(rename_city.get(city)))
        .withColumn("type", lit("mention"))
        .withColumn("mentions", get_json_object(col("entities"), "$.mentions"))
        # .withColumn("user_id2_target", col("mentions.id"))
        .select(
            col("city"),
            col("id").alias("tweet_id"),
            substring(col("created_at"),1,19).alias("created_at"),
            col("author_id").alias("user_id1_source"),
            col("user_id2_target"),
            col("type")
        )
        .withColumn("created_at", to_timestamp(col("created_at"), "yyyy-MM-dd HH:mm:ss"))
        .filter(col("user_id2_target").isNotNull())
        .limit(5)
        # # take 1000 rows for testing
        # .write
        # .mode("append")"
        # .jdbc(
        #     url=pg_url,
        #     table="tweet",
        #     properties=pg_props
        # )
    )

display(mentions.toPandas())

spark.stop()

AnalysisException: cannot resolve 'explode(get_json_object(entities, '$.mentions'))' due to data type mismatch: input to function explode should be array or map type, not string;
'Project [attachments#1981, author_created_at#1982, author_description#1983, author_entitites#1984, author_id#1985L, author_location#1986, author_name#1987, author_pinned_tweet_id#1988L, author_pm_followers_count#1989L, author_pm_following_count#1990L, author_pm_listed_count#1991L, author_pm_tweet_count#1992L, author_profile_image_url#1993, author_protected#1994, author_url#1995, author_username#1996, author_verified#1997, author_withheld#1998, context_annotations#1999, conversation_id#2000L, created_at#2001, edit_controls#2002, edit_history_tweet_ids#2003, entities#2004, ... 20 more fields]
+- Project [attachments#1981, author_created_at#1982, author_description#1983, author_entitites#1984, author_id#1985L, author_location#1986, author_name#1987, author_pinned_tweet_id#1988L, author_pm_followers_count#1989L, 